# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Neural Networks

Learn hierarchical features through layers of neurons.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

## Step 1: Neural Network Architecture

In [ ]:
# Load data
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **CRITICAL:** Neural networks require scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Neural Network for Breast Cancer Classification:")
print(f"\nInput layer:  {X_train.shape[1]} features")
print(f"Hidden layer: 100 neurons")
print(f"Output layer: 2 classes (benign or malignant)")

n_params = 30 * 100 + 100 + 100 * 2 + 2
print(f"\nTotal parameters: {n_params}")

## Step 2: Basic Multi-Layer Perceptron (MLP)

In [ ]:
# Simple MLP: one hidden layer with 100 neurons
mlp = MLPClassifier(
    hidden_layer_sizes=(100,),    # One hidden layer, 100 neurons
    activation='relu',             # ReLU activation
    max_iter=1000,                 # Max training iterations
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

train_acc = mlp.score(X_train_scaled, y_train)
test_acc = mlp.score(X_test_scaled, y_test)
auc = roc_auc_score(y_test, mlp.predict_proba(X_test_scaled)[:, 1])

print(f"MLP (1 hidden layer, 100 neurons):")
print(f"  Train accuracy: {train_acc:.4f}")
print(f"  Test accuracy:  {test_acc:.4f}")
print(f"  AUC:            {auc:.4f}")

## Step 3: Deeper Networks

In [ ]:
# Try different architectures
architectures = [
    (50,),              # 1 layer, 50 neurons
    (100,),             # 1 layer, 100 neurons
    (100, 50),          # 2 layers: 100 then 50
    (100, 100),         # 2 layers: 100 then 100
    (100, 50, 25),      # 3 layers: 100, 50, then 25
]

print(f"{'Architecture':<20} {'Train':>8} {'Test':>8}")
print("-" * 37)

for arch in architectures:
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=1000,
        random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    
    train_acc = mlp.score(X_train_scaled, y_train)
    test_acc = mlp.score(X_test_scaled, y_test)
    print(f"{str(arch):<20} {train_acc:>8.4f} {test_acc:>8.4f}")

## Step 4: Regularization (Overfitting Control)

In [ ]:
# L2 regularization (alpha parameter) prevents overfitting
alphas = [0.0, 0.0001, 0.001, 0.01, 0.1, 1.0]

print(f"{'Alpha':<8} {'Train':>8} {'Test':>8} {'Gap':>8}")
print("-" * 33)

for alpha in alphas:
    mlp = MLPClassifier(
        hidden_layer_sizes=(100,),
        activation='relu',
        alpha=alpha,  # L2 regularization strength
        max_iter=1000,
        random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    
    train_acc = mlp.score(X_train_scaled, y_train)
    test_acc = mlp.score(X_test_scaled, y_test)
    gap = train_acc - test_acc
    print(f"{alpha:<8} {train_acc:>8.4f} {test_acc:>8.4f} {gap:>8.4f}")

## Step 5: Activation Functions

In [ ]:
# Different activation functions
activations = ['identity', 'logistic', 'tanh', 'relu']

print(f"{'Activation':<15} {'Accuracy':>10}")
print("-" * 25)

for activation in activations:
    mlp = MLPClassifier(
        hidden_layer_sizes=(100,),
        activation=activation,
        max_iter=1000,
        random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    
    test_acc = mlp.score(X_test_scaled, y_test)
    print(f"{activation:<15} {test_acc:>10.4f}")

print("\nReLU is typically best for hidden layers.")

## Step 6: Training Process Monitoring

In [ ]:
# MLPClassifier can track loss during training
mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    max_iter=1000,
    random_state=42,
    warm_start=True  # Allow incremental training
)

train_losses = []

for i in range(100):  # Train for 100 epochs, track loss each time
    mlp.fit(X_train_scaled, y_train)
    train_losses.append(mlp.loss_)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(train_losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Neural Network Training Progress')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Loss decreased from {train_losses[0]:.4f} to {train_losses[-1]:.4f}")